In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Run this cell only if executing on a remote Colab kernel
if "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ:
    repo_dir = "/content/walmart-sales-forecasting"
    repo_url = "https://github.com/NikaMikeltadze/walmart-sales-forecasting.git"

    if not os.path.exists(repo_dir):
        subprocess.run(["git", "clone", repo_url, repo_dir], check=True)
    else:
        subprocess.run(["git", "-C", repo_dir, "pull"], check=True)

    # Pin everyone's Colab runtime to the same library versions as the project's requirements.txt
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", os.path.join(repo_dir, "requirements.txt"), "--quiet"],
        check=True,
    )

    os.chdir(os.path.join(repo_dir, "notebooks"))

    if repo_dir not in sys.path:
        sys.path.append(repo_dir)

    # data/raw/ is gitignored, so the clone above has code but not the competition CSVs.
    # Upload train.csv/, test.csv/, features.csv/, stores.csv, sampleSubmission.csv to this
    # folder in your Google Drive first: MyDrive/walmart-sales-forecasting/data/raw
    from google.colab import drive

    drive.mount("/content/drive")

    drive_data_dir = Path("/content/drive/MyDrive/walmart-sales-forecasting/data/raw")
    local_data_dir = Path(repo_dir) / "data" / "raw"

    if not (local_data_dir / "train.csv").exists():
        if not drive_data_dir.exists():
            raise FileNotFoundError(
                f"Expected competition data in Google Drive at {drive_data_dir}. "
                "Upload train.csv/, test.csv/, features.csv/, stores.csv, and "
                "sampleSubmission.csv there, then re-run this cell."
            )
        local_data_dir.mkdir(parents=True, exist_ok=True)
        for item in drive_data_dir.iterdir():
            dest = local_data_dir / item.name
            if dest.exists():
                continue
            if item.is_dir():
                shutil.copytree(item, dest)
            else:
                shutil.copy2(item, dest)

# ARIMA / SARIMA - Walmart Store Sales Forecasting

Person A track, third of three deliverables (XGBoost, LightGBM, ARIMA/SARIMA).
This notebook is deliberately **light and theory-focused** - per the course
instructions, ARIMA is here to demonstrate understanding of classical
time-series modeling (AR / I / MA components, stationarity, ACF/PACF,
seasonality), not to be heavily tuned for WMAE.

Structure:
- A theory section on the ARIMA/SARIMA components and why seasonal terms
  (m=52) matter for weekly retail sales.
- Diagnostics on the **company-level aggregate** weekly sales and 2-3
  representative high-volume `(Store, Dept)` series: ADF stationarity test,
  ACF/PACF, seasonal decomposition (logged as MLflow artifacts).
- `pmdarima.auto_arima` fits on those series, scored with WMAE on the same
  shared time-based validation window as the other 6 models (scoped to the
  fitted series - stated as a limitation).
- A full-coverage Kaggle submission via `src.arima_forecaster.ArimaForecaster`
  (seasonal-naive lag-52 anchor + non-seasonal residual ARIMA per series), so
  ARIMA gets a comparable Kaggle score.

Reuses `src/preprocessing.py` (`load_raw_data`), `src/validation.py`
(shared CV splitter), and `src/evaluation.py` (`weighted_mae`). Logs to the
shared DagsHub MLflow server under the `ARIMA_Training` experiment.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import warnings

import joblib
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import pmdarima as pm
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller

from src.arima_forecaster import ArimaForecaster
from src.evaluation import weighted_mae
from src.preprocessing import load_raw_data
from src.validation import describe_split, expanding_window_splits

warnings.filterwarnings("ignore")  # pmdarima/statsmodels convergence chatter
pd.set_option("display.max_columns", None)

DATA_DIR = Path.cwd().parent / "data" / "raw"
SUBMISSIONS_DIR = Path.cwd().parent / "submissions"
SUBMISSIONS_DIR.mkdir(exist_ok=True)
M = 52  # weekly seasonal period (yearly seasonality)

## MLflow setup

Logs to the shared DagsHub MLflow server so runs are comparable across both
tracks (Person A and Person B). Credentials never get typed into the
notebook itself: locally from a `.env` file in the repo root (gitignored);
on Colab from **Colab Secrets** (the key icon in the left sidebar) - each
teammate adds their own `DAGSHUB_USERNAME`/`DAGSHUB_TOKEN` secret once,
tied to their own Google account, never stored in the notebook or the repo.
If authentication fails, the setup cell below raises and stops rather than
silently falling back to a local `./mlruns` store, which would desync this
notebook's runs from the rest of the team's.

###  Manual credentials override (VS Code / non-Colab-UI runtimes)

`google.colab.userdata` (Colab Secrets) can only be read from the Colab
**browser UI**. When the Colab runtime is driven from VS Code or any other
non-UI frontend it times out (`Secrets can only be fetched when running from
the Colab UI`). This cell sets the DagsHub creds directly instead, and the
credentials cell below skips `userdata` whenever these env vars are already set.

`getpass` is used so the token is never written into this committed notebook -
run the cell and paste the values when prompted. Leave a prompt blank to fall
through to Colab Secrets / `.env` below (e.g. when you *are* on the Colab UI).

In [ ]:
import os
from getpass import getpass

# Only prompt for values not already set, so re-running cells doesn't re-ask.
# Leave a prompt blank to fall through to Colab Secrets / .env in the next cell.
if not os.environ.get("MLFLOW_TRACKING_USERNAME"):
    _user = getpass("DagsHub username (blank -> use Colab Secrets / .env): ").strip()
    if _user:
        os.environ["MLFLOW_TRACKING_USERNAME"] = _user
if not os.environ.get("MLFLOW_TRACKING_PASSWORD"):
    _token = getpass("DagsHub token (blank -> use Colab Secrets / .env): ").strip()
    if _token:
        os.environ["MLFLOW_TRACKING_PASSWORD"] = _token

In [ ]:
def _load_env_file(path):
    """Minimal .env loader (KEY=VALUE per line) - avoids adding python-dotenv
    as a dependency for a two-line credentials file. Uses setdefault so an
    env var already set in the shell (local dev) is never overridden."""
    path = Path(path)
    if not path.exists():
        return
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        os.environ.setdefault(key.strip(), value.strip())


if os.environ.get("MLFLOW_TRACKING_USERNAME") and os.environ.get("MLFLOW_TRACKING_PASSWORD"):
    # Already provided by the manual-override cell above (e.g. driving a Colab
    # runtime from VS Code, where google.colab.userdata would time out). Do NOT
    # evaluate userdata.get(...) in that case - it blocks and then raises.
    pass
elif "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ:
    # Colab Secrets (key icon, left sidebar) - each teammate adds their own
    # DAGSHUB_USERNAME/DAGSHUB_TOKEN secret, tied to their own Google account.
    # Never stored in the notebook or repo, unlike a hardcoded token would be.
    from google.colab import userdata

    os.environ["MLFLOW_TRACKING_USERNAME"] = userdata.get("DAGSHUB_USERNAME")
    os.environ["MLFLOW_TRACKING_PASSWORD"] = userdata.get("DAGSHUB_TOKEN")
else:
    _load_env_file(Path.cwd().parent / ".env")

In [ ]:
MLFLOW_TRACKING_URI = "https://dagshub.com/NikaMikeltadze/walmart-sales-forecasting.mlflow"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

try:
    mlflow.set_experiment("ARIMA_Training")
    mlflow.MlflowClient().search_experiments(max_results=1)  # force a network round trip now
except Exception as e:
    raise RuntimeError(
        "Could not authenticate to the DagsHub MLflow server at "
        f"{MLFLOW_TRACKING_URI}. Set MLFLOW_TRACKING_USERNAME and "
        "MLFLOW_TRACKING_PASSWORD (a DagsHub access token) as environment "
        "variables, then re-run this cell. Not falling back to local "
        "./mlruns - that would desync from Person B's runs."
    ) from e

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Active experiment:", mlflow.get_experiment_by_name("ARIMA_Training").name)

## Load raw data

In [ ]:
raw = load_raw_data(DATA_DIR)
for name, df in raw.items():
    print(f"{name}: {df.shape}")

assert raw["train"].shape[0] == 421_570, "unexpected train row count - check data/raw"
assert raw["test"].shape[0] > 0
assert raw["stores"].shape[0] == 45

## ARIMA / SARIMA theory

**ARIMA(p, d, q)** models a single time series from its own past:

- **AR(p) - autoregression.** The value depends linearly on its own `p`
  previous values: `y_t = c + phi_1 y_{t-1} + ... + phi_p y_{t-p} + e_t`.
  The **PACF** (partial autocorrelation) cutting off after lag `p` suggests
  the AR order.
- **I(d) - integration / differencing.** ARIMA assumes **stationarity**
  (constant mean/variance, autocovariance depending only on lag). Real sales
  trends are not stationary, so we difference the series `d` times
  (`y_t - y_{t-1}`) until it is. We test stationarity with the **Augmented
  Dickey-Fuller (ADF)** test: a small p-value (< 0.05) rejects the unit-root
  null, i.e. the series is stationary.
- **MA(q) - moving average.** The value depends on the last `q` forecast
  errors: `y_t = c + e_t + theta_1 e_{t-1} + ... + theta_q e_{t-q}`. The
  **ACF** (autocorrelation) cutting off after lag `q` suggests the MA order.

**SARIMA(p, d, q)(P, D, Q, m)** adds a seasonal copy of those terms at the
seasonal period `m`. For **weekly retail data the natural period is m = 52**
(one year), because Walmart sales have a strong yearly shape: a large
Thanksgiving/Christmas peak, a post-holiday trough, back-to-school, etc.
Seasonal differencing `D` at lag 52 (`y_t - y_{t-52}`) removes that yearly
pattern; seasonal AR/MA terms (`P`, `Q`) capture the leftover year-over-year
autocorrelation.

**Why m = 52 is expensive.** A seasonal ARIMA with m = 52 carries seasonal
lags out to 52 steps, and with only ~143 weeks of history (barely 2-3 yearly
cycles) it is both slow to fit and data-hungry per series. Fitting it for each
of the ~2,950 `(Store, Dept)` test series individually is impractical. So:

- For the **theory / diagnostics** below we fit full seasonal `auto_arima`
  (m = 52) on just the company aggregate and a few representative series,
  where a handful of fits is cheap and interpretable.
- For the **full submission** we use a fast, equivalent-in-spirit
  decomposition (`src.arima_forecaster.ArimaForecaster`): a seasonal-naive
  **lag-52 anchor** (last year's same week) plus a **non-seasonal ARIMA on the
  deseasonalized residual** `y_t - y_{t-52}`. This captures the yearly
  seasonality via the anchor and models the remaining stationary structure with
  a cheap ARIMA - see the "Final fit" section.

## Build the series to model

- **Company aggregate:** total weekly sales summed across all stores/depts -
  a single clean, high-signal series that shows the yearly seasonality plainly.
- **Representative series:** the 3 highest-volume `(Store, Dept)` series that
  have the full-length history, so per-series ARIMA has enough data.

A week is treated as a holiday week if any row that week is flagged
`IsHoliday` (WMAE weights holiday weeks 5x).

In [ ]:
train = raw["train"].copy()
train["Date"] = pd.to_datetime(train["Date"])

# Company-level weekly total sales (single aggregate series).
agg = train.groupby("Date")["Weekly_Sales"].sum().sort_index()

# Per-week holiday flag, shared by the aggregate scoring below.
holiday_by_date = train.groupby("Date")["IsHoliday"].any()

# 3 representative high-volume (Store, Dept) series with full-length history.
series_len = train.groupby(["Store", "Dept"])["Date"].nunique()
full_len = series_len.max()
eligible = series_len[series_len == full_len].index
series_means = train.groupby(["Store", "Dept"])["Weekly_Sales"].mean()
rep_keys = series_means.loc[eligible].sort_values(ascending=False).head(3).index.tolist()
rep_keys = [(int(s), int(d)) for s, d in rep_keys]


def build_series(store, dept):
    """Return (values Series, holiday Series) both indexed by Date for one series."""
    g = train[(train["Store"] == store) & (train["Dept"] == dept)].sort_values("Date")
    values = g.set_index("Date")["Weekly_Sales"]
    values = values[~values.index.duplicated(keep="last")]
    holiday = g.groupby("Date")["IsHoliday"].any().reindex(values.index).fillna(False)
    return values, holiday


print("Aggregate series:", agg.shape[0], "weeks,",
      agg.index.min().date(), "->", agg.index.max().date())
print("Representative (Store, Dept) series:", rep_keys)
agg.tail(3)

## Stationarity + ACF/PACF diagnostics

ADF tests on the aggregate (raw, first-differenced, and seasonally
differenced at lag 52), then ACF/PACF of the seasonally differenced series
and a full seasonal decomposition. All plots are saved under `submissions/`
with a leading underscore (same convention as the tree models' feature
importance plots) and attached to the `ARIMA_Aggregate` MLflow run below.

In [ ]:
def adf_report(name, series):
    series = pd.Series(series).dropna()
    stat, pval, *_ = adfuller(series, autolag="AIC")
    verdict = "stationary" if pval < 0.05 else "non-stationary"
    print(f"ADF [{name:>28}]: stat={stat:7.3f}, p-value={pval:.4f} -> {verdict} at 5%")
    return pval


adf_report("aggregate raw", agg)
adf_report("aggregate first-diff", agg.diff())
adf_report("aggregate seasonal-diff (52)", agg - agg.shift(M))

# ACF / PACF of the seasonally differenced aggregate (what the residual ARIMA sees).
agg_sdiff = (agg - agg.shift(M)).dropna()
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(agg_sdiff, ax=axes[0], lags=40)
plot_pacf(agg_sdiff, ax=axes[1], lags=40, method="ywm")
axes[0].set_title("ACF - aggregate, seasonally differenced (lag 52)")
axes[1].set_title("PACF - aggregate, seasonally differenced (lag 52)")
plt.tight_layout()
ACF_PACF_PATH = SUBMISSIONS_DIR / "_arima_acf_pacf.png"
fig.savefig(ACF_PACF_PATH)
plt.show()

# Seasonal decomposition (yearly period = 52 weeks).
decomp = seasonal_decompose(agg, period=M, model="additive")
fig = decomp.plot()
fig.set_size_inches(11, 8)
fig.suptitle("Aggregate weekly sales - seasonal decomposition (period=52)", y=1.01)
plt.tight_layout()
DECOMP_PATH = SUBMISSIONS_DIR / "_arima_decomposition.png"
fig.savefig(DECOMP_PATH, bbox_inches="tight")
plt.show()

## Time-based validation window

Uses the shared `expanding_window_splits` (same splitter as all other models)
so WMAE is comparable. The last fold is used as the reported holdout.

**Known limitation (same as the tree models):** train ends 2012-11-01, so no
validation fold can contain a Thanksgiving or Christmas week - those only
appear in the real `test.csv` horizon. WMAE weights holiday weeks 5x and those
are the two biggest sales holidays, so holdout WMAE is optimistic relative to
the true 39-week horizon.

In [ ]:
splits = expanding_window_splits(pd.Series(agg.index), n_splits=3, val_weeks=13)
print(f"{len(splits)} folds")
for i, split in enumerate(splits):
    print(f"fold {i}:", describe_split(split))

HOLDOUT = splits[-1]
print("\nReported holdout:", describe_split(HOLDOUT))

## MLflow run helper

`log_arima_run` fits a `pmdarima.auto_arima` on the train portion of a
Date-indexed series, forecasts the holdout window, scores WMAE + MAE, and logs
one flat MLflow run (params = selected order / seasonal order / AIC / CV
strategy; metrics = WMAE, MAE; artifacts = a forecast-vs-actual plot plus any
extra diagnostic images). Same flat-run + `experiment_group` tag convention as
the XGBoost/LightGBM notebooks.

In [ ]:
def _slice(series, split):
    train_s = series[series.index <= split.train_end]
    if split.train_start is not None:
        train_s = train_s[train_s.index >= split.train_start]
    val_s = series[(series.index >= split.val_start) & (series.index <= split.val_end)]
    return train_s, val_s


def log_arima_run(run_name, experiment_group, series, holiday_series, split,
                  seasonal=True, m=M, extra_params=None, extra_artifacts=None):
    """Fit auto_arima on the split's train window, score the val window, log one run."""
    train_s, val_s = _slice(series, split)

    model = pm.auto_arima(
        train_s.to_numpy(),
        seasonal=seasonal,
        m=m if seasonal else 1,
        D=1 if seasonal else None,
        max_P=1, max_Q=1, max_p=3, max_q=3,
        stepwise=True, suppress_warnings=True, error_action="ignore",
    )
    preds = np.asarray(model.predict(n_periods=len(val_s)), dtype=float)
    val_holiday = holiday_series.reindex(val_s.index).fillna(False).to_numpy()
    wmae = weighted_mae(val_s.to_numpy(), preds, val_holiday)
    mae = float(np.mean(np.abs(val_s.to_numpy() - preds)))

    with mlflow.start_run(run_name=run_name):
        mlflow.set_tag("experiment_group", experiment_group)
        mlflow.log_param("model_family", "SARIMA" if seasonal else "ARIMA")
        mlflow.log_param("seasonal", seasonal)
        mlflow.log_param("m", m if seasonal else 1)
        mlflow.log_param("order", str(model.order))
        mlflow.log_param("seasonal_order", str(model.seasonal_order))
        mlflow.log_param("aic", round(float(model.aic()), 2))
        mlflow.log_param("n_train_obs", int(len(train_s)))
        mlflow.log_param("cv_strategy", "expanding_window")
        mlflow.log_params(describe_split(split))
        if extra_params:
            mlflow.log_params(extra_params)

        mlflow.log_metric("wmae", wmae)
        mlflow.log_metric("mae", mae)

        fig, ax = plt.subplots(figsize=(11, 4))
        ax.plot(series.index, series.to_numpy(), color="0.6", label="history")
        ax.plot(val_s.index, val_s.to_numpy(), marker="o", label="actual (val)")
        ax.plot(val_s.index, preds, marker="x", label="forecast")
        ax.axvline(split.train_end, color="k", ls="--", lw=1)
        ax.set_title(f"{run_name}: order={model.order} seasonal={model.seasonal_order} WMAE={wmae:,.0f}")
        ax.legend()
        plt.tight_layout()
        plot_path = SUBMISSIONS_DIR / f"_arima_forecast_{run_name}.png"
        fig.savefig(plot_path)
        plt.show()
        mlflow.log_artifact(str(plot_path))
        for art in (extra_artifacts or []):
            mlflow.log_artifact(str(art))

    print(f"{run_name}: order={model.order} seasonal_order={model.seasonal_order} "
          f"WMAE={wmae:,.2f} MAE={mae:,.2f}")
    return {"model": model, "wmae": wmae, "mae": mae}

## Experiment 1: SARIMA on the company aggregate

Full seasonal `auto_arima` (m=52) on the aggregate series. Logs the selected
order, AIC, and holdout WMAE, and attaches the ACF/PACF and decomposition
diagnostics from above. Note the aggregate WMAE is on **summed** company sales
(tens of millions per week), so it is much larger in absolute terms than the
per-series WMAE the tree models report - the representative-series run below is
the comparable-scale number.

In [ ]:
agg_result = log_arima_run(
    "ARIMA_Aggregate", "aggregate", agg, holiday_by_date, HOLDOUT,
    seasonal=True, m=M,
    extra_params={"scope": "company_aggregate"},
    extra_artifacts=[ACF_PACF_PATH, DECOMP_PATH],
)

## Experiment 2: SARIMA on representative high-volume series

Seasonal `auto_arima` (m=52) on each of the 3 representative `(Store, Dept)`
series. These WMAE values are at the **same per-series scale** as the tree
models, but note the **limitation**: they are scoped to only these 3 series
(not all ~2,950), so they are illustrative of ARIMA's per-series behavior, not
a full-panel score. The full-panel number is the Kaggle submission below.

In [ ]:
rep_results = []
for store, dept in rep_keys:
    values, holiday = build_series(store, dept)
    res = log_arima_run(
        f"ARIMA_Series_{store}_{dept}", "representative_series",
        values, holiday, HOLDOUT, seasonal=True, m=M,
        extra_params={"store": store, "dept": dept, "scope": "single_series"},
    )
    rep_results.append(((store, dept), res))

rep_wmae_mean = float(np.mean([r["wmae"] for _, r in rep_results]))
print(f"\nMean WMAE across {len(rep_results)} representative series: {rep_wmae_mean:,.2f}")

## Final fit + full-coverage submission

`src.arima_forecaster.ArimaForecaster` turns ARIMA into a full 115,064-row
submission without fitting a slow seasonal m=52 model per series. For each
`(Store, Dept)` test series it forecasts:

> **seasonal-naive anchor** (last year's same week, `y_{t-52}`) **+
> non-seasonal `auto_arima` on the deseasonalized residual** `y_t - y_{t-52}`,
> clipped at 0.

The lag-52 anchor always reaches back into the training period for every test
week (test is <=39 weeks past train's end), so it is always defined; the cheap
non-seasonal ARIMA models the leftover stationary structure. Series that are
too short, or where auto_arima fails, fall back to the pure seasonal-naive
anchor; unseen `(Store, Dept)` pairs fall back to store/global means. This
differs from the seasonal `auto_arima` used in the theory runs above - it is
the fast, full-panel equivalent (documented as the class's design).

`fit()` just caches per-series history (cheap). `predict()` does the per-series
ARIMA fitting and is the expensive step (a few minutes over ~2,950 series).

In [ ]:
# n_jobs=-1 fans the ~2,950 per-series ARIMA fits across all CPU cores (this is
# pure CPU work - no GPU benefit, and Colab's free tier has fewer cores than a
# typical laptop, so it is not faster there). Set n_jobs=1 to debug serially.
forecaster = ArimaForecaster(m=M, min_history=104, residual_arima=True, n_jobs=-1, verbose=True)
forecaster.fit(raw["train"])
print("ArimaForecaster fit on", raw["train"].shape[0], "rows;",
      len(forecaster.history_), "series cached.")

MODEL_PATH = SUBMISSIONS_DIR / "_arima_forecaster.joblib"
joblib.dump(forecaster, MODEL_PATH)

with mlflow.start_run(run_name="ARIMA_Final") as final_run:
    mlflow.set_tag("experiment_group", "final")
    mlflow.log_param("strategy", "seasonal_naive_lag52 + residual_auto_arima")
    mlflow.log_param("m", M)
    mlflow.log_param("min_history", 104)
    mlflow.log_param("residual_arima", True)
    mlflow.log_param("max_p", forecaster.max_p)
    mlflow.log_param("max_q", forecaster.max_q)
    mlflow.log_param("max_d", forecaster.max_d)
    mlflow.log_param("n_jobs", forecaster.n_jobs)
    mlflow.log_param("n_series", len(forecaster.history_))
    # Reference metrics from the theory runs (different scopes - see notes above).
    mlflow.log_metric("aggregate_holdout_wmae", agg_result["wmae"])
    mlflow.log_metric("representative_series_wmae_mean", rep_wmae_mean)
    mlflow.log_artifact(str(MODEL_PATH))
    final_run_id = final_run.info.run_id

print("Logged ARIMA_Final, run_id:", final_run_id)

## Submission

Runs the fitted forecaster directly on raw `test.csv`-shaped data and writes
`submissions/arima_submission.csv` in the standard `Id` = `Store_Dept_Date`
format, then verifies it matches `sampleSubmission.csv` row-for-row and
attaches it to the `ARIMA_Final` run.

In [ ]:
test_preds = forecaster.predict(raw["test"])
assert not np.isnan(test_preds).any(), "forecaster produced NaN predictions on raw test data"

submission = pd.DataFrame(
    {
        "Id": (
            raw["test"]["Store"].astype(str)
            + "_"
            + raw["test"]["Dept"].astype(str)
            + "_"
            + raw["test"]["Date"].dt.strftime("%Y-%m-%d")
        ),
        "Weekly_Sales": test_preds,
    }
)

SUBMISSION_PATH = SUBMISSIONS_DIR / "arima_submission.csv"
submission.to_csv(SUBMISSION_PATH, index=False)
print("Wrote", SUBMISSION_PATH, submission.shape)

# Hard integrity checks (do not depend on any other file).
assert submission.shape == (115064, 2), submission.shape
assert list(submission.columns) == ["Id", "Weekly_Sales"]
assert not submission["Weekly_Sales"].isna().any()

# Optional row-for-row cross-check against sampleSubmission. Wrapped so a locked
# file (e.g. open in Excel) does not break the run - the Id order is already
# guaranteed by construction (same raw test frame the tree notebooks used).
try:
    sample = pd.read_csv(DATA_DIR / "sampleSubmission.csv")
    assert submission["Id"].tolist() == sample["Id"].tolist(), "Id order mismatch vs sampleSubmission"
    print("Submission matches sampleSubmission row-for-row:", submission.shape[0], "rows, no NaNs.")
except PermissionError:
    print("Note: sampleSubmission.csv is locked (open elsewhere) - skipped the row-for-row "
          "cross-check. Shape/column/NaN checks passed:", submission.shape)
submission.head()

In [ ]:
with mlflow.start_run(run_id=final_run_id):
    mlflow.log_artifact(str(SUBMISSION_PATH))

print("Submission logged as an artifact on ARIMA_Final (run_id:", final_run_id, ")")